# RSNA Pneumonia — Round 0 共用 Baseline

**這份 Notebook 是五個人共用的量尺。Round 0 請完全照抄 CONFIG 區的值，不要改任何東西。**

Round 0 的目的**不是拿高分**，是確認五個人的管線都算出同一個答案。
跑完最後一個格子會印出一行回報字串，貼到群組，五個人互相核對：

- **`checksum` 必須完全相同** —— 不同就是有人改了 `SEED` 或 `SUBSET_PATIENTS`
- **`mAP` 差距在 ±0.02 以內**算一致 —— 超過就是那個人有 bug，先修好再往下

### 跑之前要做的事

1. 到競賽頁按 **Join Competition** 接受規則，否則讀不到資料
   <https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge>
2. 右側 **Settings → Accelerator** 選 `GPU T4 x2` 或 `GPU P100`
3. 右側 **Settings → Internet** 要 **開啟**（裝 ultralytics 需要；要先做手機驗證）
4. 右側 **Input → Add Input** 搜尋 `rsna-pneumonia-detection-challenge` 掛上競賽資料
5. 只改下面 CONFIG 區的 `MY_NAME`，然後 **Run All**

T4 上大約 40~60 分鐘跑完。

In [ ]:
# =============================================================================
#  CONFIG —— 只有這一區可以改
#
#  Round 0：除了 MY_NAME 以外，全部照抄，五個人都不要動。
#  Round 1：回到 docs/02_分工與時程.md 第 1 節，只改你負責的那一個參數。
# =============================================================================
MY_NAME         = "請填你的名字"   # <<< 改這裡
ROUND           = "round0"

# --- 資料切分（Round 1 也不要改，改了就不能跟別人比） -------------------------
SEED            = 42
SUBSET_PATIENTS = 8000      # Round 0 固定 8000。設成 None 代表用全部 26684 張
VAL_RATIO       = 0.2

# --- 模型與訓練 ---------------------------------------------------------------
MODEL_NAME      = "yolov8n.pt"   # 「模型大小」負責人 Round 1 改這裡：yolov8s.pt / yolov8m.pt
IMG_SIZE        = 512            # 「影像尺寸」負責人 Round 1 改這裡：640
EPOCHS          = 10
BATCH           = 16
MOSAIC          = 1.0            # 「資料增強」負責人 Round 1 改這裡：0.0 = 關掉 mosaic

# --- 後處理 -------------------------------------------------------------------
CONF_THRESHOLD  = 0.35           # 「後處理」負責人 Round 1 改這裡
BOX_SHRINK      = 1.0            # 「後處理」負責人 Round 1 改這裡：0.85 ~ 0.90

# --- 路徑（Kaggle 固定，不用改） ----------------------------------------------
RAW  = "/kaggle/input/rsna-pneumonia-detection-challenge"
WORK = "/kaggle/working"
OUT  = f"{WORK}/dataset"

## 1. 安裝與下載共用的 metric.py

`metric.py` 是五個人**共同的量尺**，直接從我們的 GitHub repo 抓，
不要自己寫一份 —— 不同的 mAP 算法會給出不同數字，比了等於沒比。

In [ ]:
import subprocess, sys, os, json, hashlib, shutil, random
from pathlib import Path

# ultralytics 固定版本，確保五個人的訓練行為一致
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.0"], check=True)

# 共用量尺：直接抓 repo 裡的 src/metric.py
METRIC_URL = "https://raw.githubusercontent.com/Smaxoi/-RSNA-Pneumonia/main/src/metric.py"
subprocess.run(["wget", "-q", "-O", f"{WORK}/metric.py", METRIC_URL], check=True)
sys.path.insert(0, WORK)

import numpy as np
import pandas as pd
import pydicom
import cv2
import yaml
from sklearn.model_selection import train_test_split

from metric import mean_average_precision, sweep_confidence, shrink_box
import ultralytics

print("ultralytics", ultralytics.__version__)
print("metric.py 下載完成，IoU 門檻 =", __import__("metric").IOU_THRESHOLDS)

## 2. 讀標註並切 train / val，印出 checksum

**這一格的輸出要跟其他四個人核對。**

切分方式：先依 `patientId` 去重（同一個病人的多個框算一張影像），
再依三類別（Normal / No Lung Opacity / Lung Opacity）做分層抽樣，
`random_state` 固定 42。同樣的程式 + 同樣的種子 = 同樣的 val 集。

In [ ]:
labels  = pd.read_csv(f"{RAW}/stage_2_train_labels.csv")
classes = pd.read_csv(f"{RAW}/stage_2_detailed_class_info.csv")

# 一個 patientId 一列
per_image = (labels.groupby("patientId")
             .agg(n_boxes=("x", lambda s: int(s.notna().sum())), Target=("Target", "max"))
             .reset_index())
per_image = (per_image
             .merge(classes.drop_duplicates("patientId")[["patientId", "class"]],
                    on="patientId", how="left")
             .sort_values("patientId")          # 明確排序，確保決定性
             .reset_index(drop=True))

print(f"全部影像數 = {len(per_image)}")

# Round 0 用子集加快速度；抽樣也是分層 + 固定種子
if SUBSET_PATIENTS and SUBSET_PATIENTS < len(per_image):
    per_image, _ = train_test_split(per_image,
                                    train_size=SUBSET_PATIENTS,
                                    random_state=SEED,
                                    stratify=per_image["class"])
    per_image = per_image.sort_values("patientId").reset_index(drop=True)

train_ids, val_ids = train_test_split(per_image["patientId"],
                                      test_size=VAL_RATIO,
                                      random_state=SEED,
                                      stratify=per_image["class"])
val_set = set(val_ids)
per_image["split"] = np.where(per_image.patientId.isin(val_set), "val", "train")

# ---- checksum：五個人必須一模一樣 ----
val_sorted = sorted(val_set)
CHECKSUM = hashlib.md5("|".join(val_sorted).encode()).hexdigest()[:12]

print(f"本次使用  = {len(per_image)}  (train {len(per_image) - len(val_set)} / val {len(val_set)})")
print(per_image["class"].value_counts().to_string())
print()
print(f"val checksum = {CHECKSUM}   (n_val = {len(val_set)})")
print(">>> 把這串 checksum 貼到群組，五個人必須完全相同 <<<")

## 3. DICOM → PNG，並寫出 YOLO 格式標註

YOLO 要的標註是**正規化**的 `class cx cy w h`（0~1），
而競賽給的是**絕對像素**的 `x y w h`（左上角 + 寬高，1024 尺度）。轉換要小心，
這裡最容易出錯 —— 如果你的 mAP 跟別人差很多，第一個該懷疑的就是這格。

沒有肺炎的影像會寫出**空的 .txt**，這很重要：
它們是 YOLO 的 hard negative，教模型「這種片子不要畫框」。

In [ ]:
for sp in ("train", "val"):
    os.makedirs(f"{OUT}/images/{sp}", exist_ok=True)
    os.makedirs(f"{OUT}/labels/{sp}", exist_ok=True)

ORIG = 1024.0
boxes_by_pid = {}
for pid, g in labels.dropna(subset=["x"]).groupby("patientId"):
    boxes_by_pid[pid] = g[["x", "y", "width", "height"]].values.tolist()

def process(row):
    pid, sp = row
    ds = pydicom.dcmread(f"{RAW}/stage_2_train_images/{pid}.dcm")
    img = ds.pixel_array
    if IMG_SIZE != int(ORIG):
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    cv2.imwrite(f"{OUT}/images/{sp}/{pid}.png", img)

    lines = []
    for x, y, w, h in boxes_by_pid.get(pid, []):
        cx, cy = (x + w / 2) / ORIG, (y + h / 2) / ORIG
        nw, nh = w / ORIG, h / ORIG
        cx, cy = min(max(cx, 0.0), 1.0), min(max(cy, 0.0), 1.0)
        nw, nh = min(nw, 1.0), min(nh, 1.0)
        lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    # 沒有框 -> 空檔案（hard negative），不要跳過不寫
    Path(f"{OUT}/labels/{sp}/{pid}.txt").write_text("\n".join(lines))

rows = list(per_image[["patientId", "split"]].itertuples(index=False, name=None))

from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=8) as ex:
    for i, _ in enumerate(ex.map(process, rows), 1):
        if i % 2000 == 0:
            print(f"  {i}/{len(rows)}")

print("轉換完成")
print("train 影像", len(os.listdir(f'{OUT}/images/train')),
      "/ val 影像", len(os.listdir(f'{OUT}/images/val')))

## 4. 訓練

In [ ]:
data_yaml = {
    "path": OUT,
    "train": "images/train",
    "val": "images/val",
    "names": {0: "opacity"},
}
Path(f"{OUT}/data.yaml").write_text(yaml.safe_dump(data_yaml, sort_keys=False))

from ultralytics import YOLO

model = YOLO(MODEL_NAME)
model.train(
    data=f"{OUT}/data.yaml",
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    mosaic=MOSAIC,
    seed=SEED,
    deterministic=True,
    project=f"{WORK}/runs",
    name=f"{ROUND}_{MODEL_NAME.replace('.pt','')}_{IMG_SIZE}",
    exist_ok=True,
    verbose=True,
)
best = model.trainer.best
print("最佳權重：", best)

## 5. 在驗證集上用「官方 metric」算分

注意這裡**不是**看 YOLO 自己印的 mAP50 —— 那是 COCO 式的指標，跟競賽的算法不一樣。
我們用 `metric.py`，它實作了競賽的三個特殊規則（分母含 FN、無 GT 畫框直接 0 分、
兩邊都空不計分）。

推論時先用很低的門檻 `conf=0.01` 把原始框全抓出來，
之後再在本地套 `CONF_THRESHOLD` 篩 —— 這樣「後處理」負責人就能掃門檻而不用重跑推論。

In [ ]:
SCALE = ORIG / IMG_SIZE   # 把預測框還原回 1024 尺度

best_model = YOLO(best)
val_files = [f"{OUT}/images/val/{p}.png" for p in val_sorted]

raw_pred = {}
B = 64
for i in range(0, len(val_files), B):
    chunk = val_files[i:i + B]
    results = best_model.predict(chunk, imgsz=IMG_SIZE, conf=0.01, iou=0.45, verbose=False)
    for f, r in zip(chunk, results):
        pid = Path(f).stem
        bx, sc = [], []
        for b, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
            x1, y1, x2, y2 = b * SCALE
            bx.append([float(x1), float(y1), float(x2 - x1), float(y2 - y1)])
            sc.append(float(s))
        # 依信心排序（metric 需要）
        order = np.argsort(sc)[::-1]
        raw_pred[pid] = ([bx[j] for j in order], [sc[j] for j in order])
    if (i // B) % 5 == 0:
        print(f"  推論 {min(i + B, len(val_files))}/{len(val_files)}")

gt = {pid: boxes_by_pid.get(pid, []) for pid in val_sorted}

# 套用 CONF_THRESHOLD 與 BOX_SHRINK
pred = {}
for pid, (bx, sc) in raw_pred.items():
    kb, ks = [], []
    for b, s in zip(bx, sc):
        if s < CONF_THRESHOLD:
            continue
        kb.append(shrink_box(b, BOX_SHRINK) if BOX_SHRINK != 1.0 else b)
        ks.append(s)
    pred[pid] = (kb, ks)

res = mean_average_precision(gt, pred)
print()
print(f"官方 metric  mAP = {res['mAP']:.4f}   計分 {res['n_scored']} 張 / 略過 {res['n_skipped']} 張")

## 6. 回報行 —— 把這一行貼到群組

五個人互相核對：**checksum 要完全相同，mAP 差距在 ±0.02 以內算一致。**
超過 ±0.02 就是那個人有 bug，先一起看他的 CONFIG 跟第 3 格的標註轉換。

In [ ]:
line = (f"[{MY_NAME}] {ROUND} "
        f"{MODEL_NAME.replace('.pt','')} imgsz={IMG_SIZE} ep={EPOCHS} seed={SEED} "
        f"conf={CONF_THRESHOLD} shrink={BOX_SHRINK} "
        f"-> val mAP={res['mAP']:.4f}  checksum={CHECKSUM}")
print(line)

## 7.（只有「後處理」負責人要跑）信心門檻掃描

這格不用重新訓練，直接用上面的 `raw_pred` 掃一遍門檻。
掃出來的曲線就是簡報 **P1** 要用的圖 —— 它直接證明了
「這個比賽不能亂畫框」：門檻太低時 mAP 會崩掉，因為無病灶的片子畫了框就是 0 分。

In [ ]:
import matplotlib.pyplot as plt

for shrink in (1.0, 0.90, 0.85):
    curve = sweep_confidence(gt, raw_pred, thresholds=np.arange(0.05, 0.96, 0.05),
                             box_shrink=shrink)
    xs = [c for c, _ in curve]
    ys = [m for _, m in curve]
    plt.plot(xs, ys, marker="o", label=f"shrink={shrink}")
    best_c, best_m = max(curve, key=lambda t: t[1])
    print(f"shrink={shrink}:  最佳 conf={best_c:.2f}  mAP={best_m:.4f}")

plt.xlabel("confidence threshold")
plt.ylabel("official mAP")
plt.title("mAP vs confidence threshold")
plt.legend()
plt.grid(alpha=.3)
plt.show()